In [1]:
import cv2
import numpy as np
import os
import json
import random
import math
import mediapipe as mp
import pandas as pd
from datetime import datetime
from scipy.interpolate import interp1d
from tqdm.notebook import tqdm

# Configuration Constants - CHỈ DÙNG BÀN TAY
N_HAND_LANDMARKS = 21
N_TOTAL_LANDMARKS = N_HAND_LANDMARKS * 2 # 42 points in total (Left + Right)

SEQUENCE_LENGTH = 60
NUM_SAMPLES_TO_GENERATE = 100
MAX_AUGS_PER_SAMPLE = 5

# Directories (Giữ nguyên của bạn)
DATA_PATH = os.path.join('D:\\sign_language_project\\hand-sign-recognition\\data\\Processed')
DATASET_PATH = os.path.join('D:\\sign_language_project\\hand-sign-recognition\\data\\raw\\Dataset')
LOG_PATH = os.path.join('D:\\sign_language_project\\hand-sign-recognition\\data\\Logs')

os.makedirs(DATA_PATH, exist_ok=True)
os.makedirs(LOG_PATH, exist_ok=True)

class GetTime:
    def __init__(self):
        self.starttime = datetime.now()

    def get_time(self):
        return datetime.now() - self.starttime

def create_action_folder(data_path, action):
    action_path = os.path.join(data_path, action)
    os.makedirs(action_path, exist_ok=True)
    return action_path

In [ ]:
def scale_keypoints_sequence(keypoints_sequence, scale_factor_range=(0.7, 1.26), num_total_landmarks=N_TOTAL_LANDMARKS, normalize_to_01=True):
    processed_sequence = []
    if not keypoints_sequence: return processed_sequence
    current_scale_factor = random.uniform(scale_factor_range[0], scale_factor_range[1])
    
    if current_scale_factor <= 0: return [kp.copy() if isinstance(kp, np.ndarray) else kp for kp in keypoints_sequence]

    for frame_keypoints_flat in keypoints_sequence:
        if frame_keypoints_flat is None:
            processed_sequence.append(None) 
            continue
        try:
            current_points_3d = frame_keypoints_flat.copy().reshape(num_total_landmarks, 3)
        except ValueError:
            processed_sequence.append(frame_keypoints_flat.copy())
            continue

        valid_center_points_mask = np.any(current_points_3d != 0, axis=1)
        valid_center_points = current_points_3d[valid_center_points_mask]

        center_x, center_y = 0.0, 0.0
        can_calculate_center = False
        if valid_center_points.shape[0] > 0:
            center_x = np.median(valid_center_points[:, 0])
            center_y = np.median(valid_center_points[:, 1])
            can_calculate_center = True

        if can_calculate_center:
            if np.any(valid_center_points_mask):
                x_trans = current_points_3d[valid_center_points_mask, 0] - center_x
                y_trans = current_points_3d[valid_center_points_mask, 1] - center_y
                current_points_3d[valid_center_points_mask, 0] = (x_trans * current_scale_factor) + center_x
                current_points_3d[valid_center_points_mask, 1] = (y_trans * current_scale_factor) + center_y

        if normalize_to_01:
            valid_xy_mask = np.any(current_points_3d[:, :2] != 0, axis=1)
            if np.any(valid_xy_mask):
                x_coords = current_points_3d[valid_xy_mask, 0]
                y_coords = current_points_3d[valid_xy_mask, 1]
                min_x, max_x = np.min(x_coords), np.max(x_coords)
                min_y, max_y = np.min(y_coords), np.max(y_coords)

                if (max_x - min_x) > 1e-7: current_points_3d[valid_xy_mask, 0] = (x_coords - min_x) / (max_x - min_x)
                if (max_y - min_y) > 1e-7: current_points_3d[valid_xy_mask, 1] = (y_coords - min_y) / (max_y - min_y)

        processed_sequence.append(current_points_3d.flatten())
    return processed_sequence

def rotate_keypoints_sequence(keypoints_sequence, angle_degrees_range=(-15.0, 15.0), num_total_landmarks=N_TOTAL_LANDMARKS):
    rotated_sequence = []
    if not keypoints_sequence: return rotated_sequence
    
    angle_rad = math.radians(random.uniform(angle_degrees_range[0], angle_degrees_range[1]))
    cos_a, sin_a = math.cos(angle_rad), math.sin(angle_rad)

    for frame_flat in keypoints_sequence:
        if frame_flat is None:
            rotated_sequence.append(None)
            continue
        try:
            points = frame_flat.copy().reshape(num_total_landmarks, 3)
        except ValueError:
            rotated_sequence.append(frame_flat.copy())
            continue

        valid_mask = np.any(points != 0, axis=1)
        valid_pts = points[valid_mask]
        
        if valid_pts.shape[0] == 0:
            rotated_sequence.append(frame_flat.copy())
            continue
            
        cx, cy = np.median(valid_pts[:, 0]), np.median(valid_pts[:, 1])
        
        x_trans = points[valid_mask, 0] - cx
        y_trans = points[valid_mask, 1] - cy
        
        points[valid_mask, 0] = (x_trans * cos_a - y_trans * sin_a) + cx
        points[valid_mask, 1] = (x_trans * sin_a + y_trans * cos_a) + cy
        rotated_sequence.append(points.flatten())
        
    return rotated_sequence

def translate_keypoints_sequence(keypoints_sequence, translate_x_range=(-0.05, 0.05), translate_y_range=(-0.05, 0.05), clip_to_01=True, num_total_landmarks=N_TOTAL_LANDMARKS):
    translated_sequence = []
    if not keypoints_sequence: return translated_sequence
    dx = random.uniform(translate_x_range[0], translate_x_range[1])
    dy = random.uniform(translate_y_range[0], translate_y_range[1])

    for frame_flat in keypoints_sequence:
        if frame_flat is None:
            translated_sequence.append(None)
            continue
        try:
            points = frame_flat.copy().reshape(num_total_landmarks, 3)
        except ValueError:
            translated_sequence.append(frame_flat.copy())
            continue

        valid_mask = np.any(points != 0, axis=1)
        points[valid_mask, 0] += dx
        points[valid_mask, 1] += dy

        if clip_to_01:
            points[valid_mask, 0] = np.clip(points[valid_mask, 0], 0.0, 1.0)
            points[valid_mask, 1] = np.clip(points[valid_mask, 1], 0.0, 1.0)

        translated_sequence.append(points.flatten())
    return translated_sequence

def time_stretch_keypoints_sequence(keypoints_sequence, speed_factor_range=(0.8, 1.2)):
    if not keypoints_sequence or all(kp is None for kp in keypoints_sequence): return keypoints_sequence
    valid_frames = [kp for kp in keypoints_sequence if kp is not None]
    if not valid_frames: return keypoints_sequence

    speed_factor = random.uniform(speed_factor_range[0], speed_factor_range[1])
    if speed_factor <= 0 or speed_factor == 1.0: return [kp.copy() for kp in valid_frames]

    num_new_frames = int(round(len(valid_frames) / speed_factor))
    if num_new_frames == 0: return [valid_frames[0].copy()]

    indices = np.clip(np.round(np.linspace(0, len(valid_frames) - 1, num_new_frames)).astype(int), 0, len(valid_frames) - 1)
    return [valid_frames[i].copy() for i in indices]



In [ ]:
def generate_augmented_samples(original_sequence, augmentation_functions, num_samples_to_generate, max_augs_per_sample=3):
    generated_samples = []
    if not original_sequence or not augmentation_functions: return generated_samples

    for _ in range(num_samples_to_generate):
        current_seq = [kp.copy() for kp in original_sequence]
        num_augs = random.randint(1, min(max_augs_per_sample, len(augmentation_functions)))
        selected_augs = random.sample(augmentation_functions, num_augs)

        for aug_func in selected_augs:
            current_seq = aug_func(current_seq)
            if not current_seq: break

        if current_seq: generated_samples.append(current_seq)

    return generated_samples

In [ ]:
def extract_keypoints_tasks(mp_image, hand_model):
    hand_result = hand_model.detect(mp_image)

    left_hand_kps = np.zeros((N_HAND_LANDMARKS, 3))
    right_hand_kps = np.zeros((N_HAND_LANDMARKS, 3))

    if hand_result.hand_landmarks:
        for idx, landmarks in enumerate(hand_result.hand_landmarks):
            handedness = hand_result.handedness[idx][0].category_name
            kps = np.array([[lm.x, lm.y, lm.z] for lm in landmarks])
            
            if handedness == 'Left':
                left_hand_kps = kps
            elif handedness == 'Right':
                right_hand_kps = kps

    return np.concatenate([left_hand_kps, right_hand_kps]).flatten()

def sequence_frames(video_path, hand_model):
    sequence = []
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if total_frames <= 0: return sequence
    step = max(1, total_frames // 100) 

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        if int(cap.get(cv2.CAP_PROP_POS_FRAMES)) % step != 0: continue

        try:
            image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)
            keypoints = extract_keypoints_tasks(mp_image, hand_model)
            sequence.append(keypoints)
        except Exception:
            continue

    cap.release()
    return sequence

def interpolate_keypoints(keypoints_sequence, target_len=SEQUENCE_LENGTH):
    """Interpolates varying length sequences to a standard 60 frames."""
    if not keypoints_sequence: return None

    original_times = np.linspace(0, 1, len(keypoints_sequence))
    target_times = np.linspace(0, 1, target_len)
    num_features = keypoints_sequence[0].shape[0]
    interpolated_sequence = np.zeros((target_len, num_features))

    for feature_idx in range(num_features):
        feature_values = [frame[feature_idx] for frame in keypoints_sequence]
        interpolator = interp1d(original_times, feature_values, kind='cubic', bounds_error=False, fill_value="extrapolate")
        interpolated_sequence[:, feature_idx] = interpolator(target_times)

    return interpolated_sequence

In [4]:
import re

def sanitize_folder_name(name):
    """
    Removes or replaces characters that are illegal in Windows file paths.
    """
    # Replace illegal characters with an underscore or empty string
    # You can customize this if you want to replace '<' with 'less_than'
    safe_name = re.sub(r'[<>:"/\\|?*]', '_', name)
    
    # Optional: Remove trailing spaces or dots which can also cause issues
    return safe_name.strip('. ')

# --- Update your create_action_folder function ---
def create_action_folder(data_path, action):
    # Sanitize the action string before joining it to the path
    safe_action = sanitize_folder_name(action)
    action_path = os.path.join(data_path, safe_action)
    os.makedirs(action_path, exist_ok=True)
    return action_path

In [ ]:
label_file = os.path.join(DATASET_PATH, 'Labels', 'label.csv')
video_folder = os.path.join(DATASET_PATH, 'Videos')

if not os.path.exists(label_file):
    print(f"ERROR: Cannot find label file at {label_file}")
else:
    df = pd.read_csv(label_file)
    selected_actions = sorted(df['LABEL'].unique())
    label_map = {action: idx for idx, action in enumerate(selected_actions)}
    
    # Save Label Map
    with open(os.path.join(LOG_PATH, 'label_map.json'), 'w', encoding='utf-8') as f:
        json.dump(label_map, f, ensure_ascii=False, indent=4)
        
    print(f"Selected {len(selected_actions)} unique actions.")
    
    augmentations = [scale_keypoints_sequence, rotate_keypoints_sequence, translate_keypoints_sequence, time_stretch_keypoints_sequence]
    timer = GetTime()
    
    BaseOptions = mp.tasks.BaseOptions
    VisionRunningMode = mp.tasks.vision.RunningMode
    hand_options = mp.tasks.vision.HandLandmarkerOptions(
        base_options=BaseOptions(model_asset_path='D:\\sign_language_project\\hand-sign-recognition\\notebooks\\models\\hand_landmarker.task'),
        running_mode=VisionRunningMode.IMAGE, num_hands=2
    )

    print(f"[{datetime.now()}] Start processing data...")

    with mp.tasks.vision.HandLandmarker.create_from_options(hand_options) as hand_model:
        for _, row in tqdm(df.iterrows(), total=len(df), desc='Processing Videos'):
            action, video_file = row['LABEL'], row['VIDEO']
            label = label_map[action]
            action_path = create_action_folder(DATA_PATH, action)
            video_path = os.path.join(video_folder, video_file)
            if not os.path.exists(video_path): continue
            
            # 1. Extract Base Sequence using Tasks API
            frame_lists = sequence_frames(video_path, hand_model)
            if not frame_lists: continue
            
            # 2. Generate Offline Augmentations
            augmenteds = generate_augmented_samples(frame_lists, augmentations, NUM_SAMPLES_TO_GENERATE, MAX_AUGS_PER_SAMPLE)
            augmenteds.append(frame_lists) # Append the original non-augmented sequence
            
            # Calculate safe starting index so it doesn't overwrite old data
            idx = len([f for f in os.listdir(action_path) if f.endswith('.npz')])
            
            # 3. Interpolate and Save to Disk
            for aug in augmenteds:
                seq = interpolate_keypoints(aug, SEQUENCE_LENGTH)
                if seq is not None:
                    np.savez(os.path.join(action_path, f'{idx}.npz'), sequence=seq, label=label)
                    idx += 1

    print(f"\n{'-'*50}\nDATA PROCESSING COMPLETED. Total Elapsed Time: {timer.get_time()}")

Selected 3315 unique actions.
[2026-06-01 11:38:34.214563] Start processing data...


Processing Videos:   0%|          | 0/4362 [00:00<?, ?it/s]


--------------------------------------------------
DATA PROCESSING COMPLETED. Total Elapsed Time: 4:38:13.497324


In [3]:
import os
import glob
import shutil
from sklearn.model_selection import train_test_split

DATA_PATH = r'D:\sign_language_project\hand-sign-recognition\data\Processed'
OUTPUT_PATH = r'D:\sign_language_project\hand-sign-recognition\data\Processed_Split'
VAL_SPLIT = 0.1
TEST_SPLIT = 0.1

file_pattern = os.path.join(DATA_PATH, '**', '*.npz')
all_files = glob.glob(file_pattern, recursive=True)

labels_for_stratify = [os.path.basename(os.path.dirname(p)) for p in all_files]

train_files, temp_files, train_labels, temp_labels = train_test_split(
    all_files, 
    labels_for_stratify,
    test_size=(VAL_SPLIT + TEST_SPLIT),  
    shuffle=True,
    random_state=42,
    stratify=labels_for_stratify 
)

val_files, test_files = train_test_split(
    temp_files,
    test_size=TEST_SPLIT / (VAL_SPLIT + TEST_SPLIT),
    shuffle=True,
    random_state=42,
    stratify=temp_labels 
)

def copy_files_to_folder(file_list, subset_name):
    for file_path in file_list:
        action_name = os.path.basename(os.path.dirname(file_path))
        file_name = os.path.basename(file_path)
        dest_dir = os.path.join(OUTPUT_PATH, subset_name, action_name)
        os.makedirs(dest_dir, exist_ok=True)
        dest_file_path = os.path.join(dest_dir, file_name)
        shutil.copy2(file_path, dest_file_path)

copy_files_to_folder(train_files, 'train')
copy_files_to_folder(val_files, 'val')
copy_files_to_folder(test_files, 'test')

In [5]:
import numpy as np

# Thay đường dẫn này bằng đường dẫn tới 1 file .npz thực tế trên máy bạn
file_path = r'D:\1.npz'

# 1. Load file npz
npz_file = np.load(file_path)

# 2. Xem các trường dữ liệu (keys) có trong file này
print("Các trường dữ liệu trong file:", npz_file.files) 
# Output kỳ vọng: ['sequence', 'label']

# 3. Lấy dữ liệu ra các biến
sequence_data = npz_file['sequence']
label_data = npz_file['label']

# 4. In thử kích thước và giá trị
print("-------------------------------------------------")
print("Kích thước ma trận sequence (Frames x Features):", sequence_data.shape)
# Output kỳ vọng: (60, 201)

print("Nhãn (ID) của video này là:", label_data)

# Nếu muốn xem thử tọa độ của frame đầu tiên (frame 0)
print("-------------------------------------------------")
print("Tọa độ của frame đầu tiên (201 chiều):")
print(sequence_data[2])

# Đóng file sau khi đọc xong (thực hành tốt để giải phóng bộ nhớ)
npz_file.close()

Các trường dữ liệu trong file: ['sequence', 'label']
-------------------------------------------------
Kích thước ma trận sequence (Frames x Features): (60, 201)
Nhãn (ID) của video này là: 0
-------------------------------------------------
Tọa độ của frame đầu tiên (201 chiều):
[ 5.17595419e-01  2.93037392e-01 -6.08063811e-01  5.31136015e-01
  2.68517260e-01 -5.71577298e-01  5.38405041e-01  2.69838658e-01
 -5.71861248e-01  5.45064845e-01  2.71793430e-01 -5.71941354e-01
  5.05920507e-01  2.69152449e-01 -5.74125062e-01  4.97703681e-01
  2.71004115e-01 -5.74242661e-01  4.89719912e-01  2.73975400e-01
 -5.74645517e-01  5.56496480e-01  3.00298761e-01 -3.46839718e-01
  4.80153717e-01  3.03313703e-01 -3.54498739e-01  5.35288651e-01
  3.35420509e-01 -5.26493310e-01  5.02181904e-01  3.39330693e-01
 -5.28309956e-01  6.15884383e-01  5.35173219e-01 -2.07627312e-01
  4.29155804e-01  5.34839098e-01 -2.25459354e-01  6.27872365e-01
  7.86367757e-01 -1.87936996e-01  4.20545762e-01  8.09385614e-01
 -1.